In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import random
from IPython.display import Audio

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset,DataLoader

from tqdm import tqdm
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kgg_key")
secret_value_1 = user_secrets.get_secret("kgg_user")
secret_value_2 = user_secrets.get_secret("WANDB_API_KEY")

DATA_ROOT = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
STEMS = {'vocals.wav','other.wav','bass.wav','drums.wav'} 
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']

# NOISE DATASET
root_dir = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/'
csv_file = '/meta/esc50.csv'
audio_folder = '/audio/'


SR = 22050
DURATION = 30
model_name = "cnn"
try:
    dir_path = f"/kaggle/working/{model_name}"
    os.makedirs(dir_path, exist_ok=True)
    print(f"Directory created at: {dir_path}")
except Exception as e:
    print(f"Error creating directory: {e}")

#============ Check for GPU availability ==============================#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    print("GPU is available.Setting RANDOM_SEED .... ")
        # setting for both CPU and GPU
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("   Running on CPU")
print("\n✅ Environment setup complete!")

import warnings
warnings.filterwarnings("ignore")

Directory created at: /kaggle/working/cnn
🚀 Using device: cuda
GPU is available.Setting RANDOM_SEED .... 
   GPU: Tesla T4
   Memory: 14.6 GB

✅ Environment setup complete!


# Definition

## 1. Utility

In [3]:
def build_dataset(root_dir,seed=42):
    """
        This takes root dirstory and load paths of all stem files as a dictionary.
        Returns a Dictionary.
    """
    global SR
    global DURATION
    LENGTH = SR*DURATION
    
    # Initialize empty dictionaries
    loaded_song = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    total_stem_file = 0
    for g in tqdm(GENRES,desc="Loading stems into dict.."):
        folder_path = os.path.join(root_dir,g)
        if os.path.exists(folder_path) and os.path.isdir(folder_path):
            for i in range(0,100):
                song_folder = g + '.' + f"{i:05d}"
                for s in STEMS:
                    stem_file_path = os.path.join(folder_path ,song_folder ,s)
                    if os.path.exists(stem_file_path):
                        total_stem_file += 1

                        y = load_and_fix(stem_file_path)
                        loaded_song[g][s.replace('.wav', '')].append(y)
                    else:
                        print(f"Stem file {s} not exists !")
            
        else:
            print(f"Folder '{g}' not exists !")

    print("✅ Total number of stems music file loaded successfully : ", total_stem_file)
    
    return loaded_song

def load_and_fix(path,sr=SR,duration=DURATION):
    LENGTH = sr*duration    
    waveform, _sr_ = torchaudio.load(path)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample if needed
    if _sr_ != sr:
        resampler = torchaudio.transforms.Resample(_sr_, sr)
        waveform = resampler(waveform)

    waveform = waveform.squeeze(0)

    # Trim or pad
    if waveform.shape[0] >= LENGTH:
        return waveform[:LENGTH]
    else:
        padding = LENGTH - waveform.shape[0]
        return torch.nn.functional.pad(waveform, (0, padding))

def load_noise_audios(root_dir,audio_folder,csv_file):
    """
     It will extract all noise audio waveform and store in a dictionary.
    """
    noise = []
    df_noise = pd.read_csv(root_dir+csv_file)
    total_noise = 0
    
    for i in tqdm(df_noise.index,desc="Loading Noise audio ... "):
        filename = df_noise.loc[i]['filename']
        path = root_dir+audio_folder+filename
        y = load_and_fix(path,sr=SR,duration=5)    
        noise.append(y)
        total_noise += 1
    noise_tensor = torch.stack([nt for nt in noise])
    print(f"Total noise files loaded : {total_noise}")
    return noise_tensor

def genre_to_idx(targets):
    genre_to_id = {'blues':0, 'classical':1, 'country':2, 'disco':3, 'hiphop':4,'jazz':5, 'metal':6, 'pop':7, 'reggae':8, 'rock':9}
    y = torch.tensor([genre_to_id[g] for g in targets])

    return y

def idx_to_genre(targets):
    id_to_genre = {0:'blues', 1:'classical', 2:'country', 3:'disco', 4:'hiphop',5:'jazz', 6:'metal', 7:'pop', 8:'reggae', 9:'rock'}
    y = [id_to_genre[id] for id in targets]

    return y    
# ====================================================== Augmentation utility ====================================== #
    
def aug_stem(stem,sr=SR,max_shift_ms=40):
    B, T = stem.shape
    device = stem.device
    # random gain
    gain = torch.rand(stem.size(0),1, device=stem.device) * 0.4 + 0.8
    stem = stem * gain
    
    #3. time shift
    max_shift = int(sr * max_shift_ms / 1000)
    shifts = torch.randint(-max_shift, max_shift+1, (B,), device=device)
    base_idx = torch.arange(T, device=stem.device).expand(B, T)
    shifted_idx = base_idx - shifts.unsqueeze(1)
    mask = (shifted_idx >= 0) & (shifted_idx < T)
    shifted_idx = shifted_idx.clamp(0, T-1)
    shifted = torch.gather(stem, 1, shifted_idx)
    shifted = shifted * mask
    #4. small noise
    noise = torch.randn_like(shifted) * 0.002
    output = shifted + noise
    return output

def add_noise(mashup,noise):
    B, T = mashup.shape
    N, L = noise.shape
    
    max_noise = 5
    
    noise_ids = torch.randint(0, N, (B, max_noise), device=mashup.device)
    start_pos = torch.randint(0, T-L, (B, max_noise), device=mashup.device)

    noise_segments = noise[noise_ids]

    offset = torch.arange(L, device=mashup.device)
    offset = offset.view(1,1,L)

    indices = start_pos.unsqueeze(-1) + offset

    flat_indices = indices.view(B, -1)
    flat_noise   = noise_segments.view(B, -1)

    mashup.scatter_add_(1, flat_indices, flat_noise)
    return mashup

In [4]:
class MelDataset(Dataset):
    def __init__(self,stem_dict,mashup_list):  # paths : List[(path,label)]
        self.stem_dict = stem_dict
        self.mashup_list = mashup_list
    def __len__(self):
        return len(self.mashup_list)
    def __getitem__(self,idx):
        v,d,b,o,l = (self.mashup_list[idx]['vocals'],
                        self.mashup_list[idx]['drums'],
                        self.mashup_list[idx]['bass'],
                        self.mashup_list[idx]['other'],
                        self.mashup_list[idx]['label'] )
        return (self.stem_dict[l]['vocals'][v],
                self.stem_dict[l]['drums'][d],
                self.stem_dict[l]['bass'][b],
                self.stem_dict[l]['other'][o],
                l )

In [5]:
sl = build_dataset(DATA_ROOT)

Loading stems into dict..: 100%|██████████| 10/10 [09:49<00:00, 58.94s/it]

✅ Total number of stems music file loaded successfully :  4000


In [6]:
noise = load_noise_audios(root_dir,audio_folder,csv_file)

Loading Noise audio ... : 100%|██████████| 2000/2000 [00:55<00:00, 35.83it/s]


Total noise files loaded : 2000


In [8]:
config = {
    "batch_size" : 64
}

mel_transform = T.MelSpectrogram(
    sample_rate = 22050,
    n_fft = 1024,
    hop_length = 1024,
    n_mels= 64
).to(device)

amplitude_to_db = T.AmplitudeToDB().to(device)

train_mashups = []
val_mashups = []
for g in GENRES:
    for i in range(5000):
        stems = {'label':g}
        for s in STEM_KEYS:
            stem_id = random.choice(range(100))
            stems[s] = stem_id
        train_mashups.append(stems)
    for j in range(200):
        val_stems = {'label':g}
        for s in STEM_KEYS:
            stem_id = random.choice(range(100))
            val_stems[s] = stem_id
        val_mashups.append(val_stems)


train_dataset = MelDataset(sl,train_mashups)
val_dataset = MelDataset(sl,val_mashups)
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

print(f"Size of Train Dataloader : {len(train_loader)}  | Size of Val Dataloader : {len(val_loader)}")
print("✅")

Size of Train Dataloader : 782  | Size of Val Dataloader : 32
✅


In [9]:
class MelCNN(nn.Module):
    def __init__(self, num_classes):
        super(MelCNN, self).__init__()
        # First convolutional block
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)

        #self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc1 = nn.Linear(64*8*80,128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):

        x = x.unsqueeze(1)  # (B,1,128,1292)

        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        #x = self.global_pool(x)
        x = x.view(x.size(0), -1)

        x = self.fc1(x)
        x = self.fc2(x)

        return x

In [10]:
def train(model,train_loader,noise,loss_fn, optimizer, num_epochs=5):
    train_loss = []
    train_f1_score = []
    model.train()
    max_f1 = 0.0
    best_model = model
    for epoch in range(num_epochs):
        running_loss = 0.0
        progress_bar = tqdm(enumerate(train_loader), total=len(train_loader),desc=f"Epoch {epoch+1}/{num_epochs}")
        for i,(vocals,drums,bass,other,label) in progress_bar:
            vocals = vocals.to(device,non_blocking=True)
            drums = drums.to(device,non_blocking=True)
            bass = bass.to(device,non_blocking=True)
            other = other.to(device,non_blocking=True)
            label = genre_to_idx(label).to(device,non_blocking=True)

            vocals = aug_stem(vocals)
            drums = aug_stem(drums)
            bass = aug_stem(bass)
            other = aug_stem(other)

            noise = noise.to(device)
            
            
            mashup = vocals+drums+bass+other
            mashup = add_noise(mashup,noise)
            
            optimizer.zero_grad()
            mels = mel_transform(mashup)
            mels = amplitude_to_db(mels)
            
            outputs = model(mels)
            loss = loss_fn(outputs, label)
        
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            avg_loss = running_loss / (i + 1)

            progress_bar.set_postfix({
                'loss': f"{avg_loss:.4f}"
            })
        loss = running_loss / len(train_loader)
        t,p,f1score = validation(model,val_loader,noise,criterion)
        if f1score > max_f1 :
            print(f"Train Loss: {loss:.4f}")
            max_f1 = f1score
            best_model = model
        else:
            print(f"Train Loss: {loss:.4f}")
            return best_model
        
        

In [11]:
def validation(model,val_loader,noise,loss_fn):
    model.eval()
    val_loss = 0.0
    all_true = []
    all_pred = []
    with torch.no_grad():
        for (vocals,drums,bass,other,label) in tqdm(val_loader, desc="Evaluating"):
            vocals = vocals.to(device,non_blocking=True)
            drums = drums.to(device,non_blocking=True)
            bass = bass.to(device,non_blocking=True)
            other = other.to(device,non_blocking=True)
            label = genre_to_idx(label).to(device,non_blocking=True)

            vocals = aug_stem(vocals)
            drums = aug_stem(drums)
            bass = aug_stem(bass)
            other = aug_stem(other)

            noise = noise.to(device)
            
            
            mashup = vocals+drums+bass+other
            mashup = add_noise(mashup,noise)
            
            mels = mel_transform(mashup)
            mels = amplitude_to_db(mels)
            
            output = model(mels)
            loss = loss_fn(output,label)

            probs = torch.softmax(output,dim=1) 
            predicted_y = torch.argmax(probs,dim=1)
            
            val_loss += loss.item()

            all_true.append(label)
            all_pred.append(predicted_y)
        all_true = torch.cat(all_true,dim=0).cpu().numpy()
        all_pred = torch.cat(all_pred,dim=0).cpu().numpy()
    
        loss = val_loss/len(val_loader)
        f1score = f1_score(all_true,all_pred,average='macro')

        print(f"Validation Loss : {loss},  F1_score : {f1score}")
        return all_true,all_pred,f1score
            

In [12]:
model = MelCNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


best_model = train(model,train_loader,noise,criterion, optimizer, num_epochs=10)

Evaluating: 100%|██████████| 32/32 [00:14<00:00,  2.24it/s]


Validation Loss : 0.38499314058572054,  F1_score : 0.8896386167092756
Train Loss: 1.9788


Evaluating: 100%|██████████| 32/32 [00:13<00:00,  2.33it/s]


Validation Loss : 0.07555990395485424,  F1_score : 0.9762162104552627
Train Loss: 0.1223


Evaluating: 100%|██████████| 32/32 [00:13<00:00,  2.39it/s]


...

Evaluating: 100%|██████████| 32/32 [00:13<00:00,  2.37it/s]


Validation Loss : 0.022041228961825254,  F1_score : 0.9919963412089856
Train Loss: 0.0309


Evaluating: 100%|██████████| 32/32 [00:13<00:00,  2.29it/s]

Validation Loss : 0.05174839614119264,  F1_score : 0.9793687735930897
Train Loss: 0.0316


MelCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=40960, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [14]:
if os.path.exists(f"/kaggle/working/{model_name}"):
    torch.save(model.state_dict(), f"/kaggle/working/{model_name}/model.pth")
    print(f"✅ {model_name} saved.")
else:
    print(f"Path not exists.")

✅ cnn saved.


In [15]:
import kagglehub

# Replace with path to directory containing model files.
LOCAL_MODEL_DIR = f'/kaggle/working/{model_name}'

MODEL_SLUG = model_name # Replace with model slug.

# Learn more about naming model variations at
# https://www.kaggle.com/docs/models#name-model.
VARIATION_SLUG = 'default' # Replace with variation slug.

kagglehub.model_upload(
  handle = f"akashkumbhakar/{MODEL_SLUG}/pyTorch/{VARIATION_SLUG}",
  local_model_dir = LOCAL_MODEL_DIR,
  version_notes = 'Update 2026-03-08')

Uploading Model https://api.kaggle.com/models/akashkumbhakar/cnn/pyTorch/default ...
Starting upload for file /kaggle/working/cnn/model.pth


Uploading: 100%|██████████| 21.1M/21.1M [00:00<00:00, 29.8MB/s]

Upload successful: /kaggle/working/cnn/model.pth (20MB)


Your model instance version has been created.
Files are being processed...
See at: https://api.kaggle.com/models/akashkumbhakar/cnn/pyTorch/default
